# 01 — Data Collection: Grab App Reviews

Scrapes Grab reviews from:
- **Google Play Store** (`com.grabtaxi.passenger`, SG market) via `google-play-scraper`
- **Apple App Store** (app ID `647268330`, SG market) via `app-store-scraper`

> **Note:** `app-store-scraper` occasionally fails if Apple's iTunes RSS endpoint returns empty responses.
> The notebook handles this gracefully and proceeds with Play Store data alone if needed.

Output: `../data/grab_reviews_raw.csv`

In [ ]:
import os, time, warnings
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Libraries imported. Output directory ready.')

## 1. Google Play Store

In [ ]:
from google_play_scraper import reviews, Sort

PLAY_APP_ID = 'com.grabtaxi.passenger'
TARGET_PLAY = 10000
BATCH_SIZE  = 200

play_reviews = []
continuation_token = None

with tqdm(total=TARGET_PLAY, desc='Google Play') as pbar:
    while len(play_reviews) < TARGET_PLAY:
        batch, continuation_token = reviews(
            PLAY_APP_ID,
            lang='en',
            country='sg',
            sort=Sort.NEWEST,
            count=BATCH_SIZE,
            continuation_token=continuation_token,
        )
        if not batch:
            print('No more reviews available from Play Store.')
            break
        play_reviews.extend(batch)
        pbar.update(len(batch))
        if continuation_token is None:
            break
        time.sleep(0.5)

print(f'Collected {len(play_reviews):,} Google Play reviews')

In [ ]:
df_play = pd.DataFrame(play_reviews)
print('Raw Play columns:', df_play.columns.tolist())

df_play = df_play.rename(columns={
    'content':  'review_text',
    'score':    'rating',
    'at':       'date',
    'userName': 'author',
})
df_play['source'] = 'google_play'
df_play = df_play[['review_text', 'rating', 'date', 'author', 'source']]
print(f'Google Play: {df_play.shape[0]:,} rows')
df_play.head(2)

## 2. Apple App Store

`app-store-scraper` relies on Apple's iTunes RSS API which can return empty responses.  
The cell below tries to scrape, inspects the actual column names returned, and falls back gracefully.

In [ ]:
from app_store_scraper import AppStore

ios_dfs = []

try:
    grab_ios = AppStore(country='sg', app_name='grab', app_id='647268330')
    grab_ios.review(how_many=3000)
    raw_reviews = grab_ios.reviews
    print(f'Fetched {len(raw_reviews):,} App Store reviews')
except Exception as e:
    raw_reviews = []
    print(f'App Store scraping failed: {e}')

if raw_reviews:
    df_ios_raw = pd.DataFrame(raw_reviews)
    print('App Store columns:', df_ios_raw.columns.tolist())

    # Build rename map dynamically from whatever columns are actually present
    candidate_text_cols  = ['review', 'body', 'content', 'text']
    candidate_score_cols = ['rating', 'score', 'stars']
    candidate_date_cols  = ['date', 'at', 'timestamp']
    candidate_user_cols  = ['userName', 'user', 'author', 'name']

    def first_match(candidates, cols):
        return next((c for c in candidates if c in cols), None)

    cols = df_ios_raw.columns.tolist()
    text_col  = first_match(candidate_text_cols,  cols)
    score_col = first_match(candidate_score_cols, cols)
    date_col  = first_match(candidate_date_cols,  cols)
    user_col  = first_match(candidate_user_cols,  cols)

    print(f'Mapped → text:{text_col}, score:{score_col}, date:{date_col}, user:{user_col}')

    rename = {}
    if text_col:  rename[text_col]  = 'review_text'
    if score_col: rename[score_col] = 'rating'
    if date_col:  rename[date_col]  = 'date'
    if user_col:  rename[user_col]  = 'author'

    df_ios = df_ios_raw.rename(columns=rename)
    for col in ['review_text', 'rating', 'date', 'author']:
        if col not in df_ios.columns:
            df_ios[col] = None
    df_ios['source'] = 'app_store'
    df_ios = df_ios[['review_text', 'rating', 'date', 'author', 'source']]
    ios_dfs.append(df_ios)
    print(f'App Store DataFrame: {df_ios.shape}')
else:
    print('No App Store reviews collected — proceeding with Google Play data only.')
    print('(This is common when Apple\'s iTunes RSS returns empty responses.)')

## 3. Combine and Save

In [ ]:
all_dfs = [df_play] + ios_dfs
df = pd.concat(all_dfs, ignore_index=True)

# Normalise date column
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
df['date'] = df['date'].dt.tz_convert(None)

print(f'Total reviews : {len(df):,}')
print(f'Date range    : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Rating dist   :\n{df["rating"].value_counts().sort_index()}')
print(f'Source split  :\n{df["source"].value_counts()}')

In [ ]:
OUT = '../data/grab_reviews_raw.csv'
df.to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows to {OUT}')
df.head()

## 4. Quick Sanity Check

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution (raw)')
axes[0].set_xlabel('Stars')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df.set_index('date').resample('ME').size().plot(ax=axes[1], color='coral')
axes[1].set_title('Review Volume by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/01_sanity_check.png', dpi=100)
plt.show()
print('Done. Scraping notebook complete.')